# 04 — Evaluation

We evaluate the fine-tuned Qwen2.5-0.5B on two benchmarks:

- **MATH test split** — in-distribution, same dataset family as training
- **GSM8K** — out-of-distribution, tests transfer to a different math benchmark

We measure **pass@1** — the model gets one shot at each problem and we check
if the extracted answer matches the gold answer.

In [1]:
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from tqdm import tqdm

## Helper Functions

In [2]:
def extract_boxed(text):
    match = re.search(r'\\boxed\{([^{}]+)\}', text)
    if match:
        clean = re.sub(r'\\text\{[^}]*\}', '', match.group(1))
        return clean.strip()
    return None

def extract_answer_tag(text):
    # Format 1: <answer>...</answer>
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if match:
        return match.group(1).strip()
    # Format 2: **Answer:** 42
    match = re.search(r'\*\*Answer:\*\*\s*(.+)', text)
    if match:
        return match.group(1).strip()
    # Format 3: \boxed{} in model output
    match = re.search(r'\\boxed\{([^{}]+)\}', text)
    if match:
        return match.group(1).strip()
    # Format 4: Answer: 42
    match = re.search(r'[Aa]nswer:\s*(.+)', text)
    if match:
        return match.group(1).strip()
    # Format 5: orphaned </answer>
    match = re.search(r'(\S+)\s*</answer>', text)
    if match:
        return match.group(1).strip()
    return None

def normalize(text):
    return (text
        .replace('$', '')
        .replace(' ', '')
        .replace('\n', '')
        .replace(r'\(', '')
        .replace(r'\)', '')
        .replace(r'\[', '')
        .replace(r'\]', '')
        .replace(r'\boxed{', '')
        .replace('}', '')
        .strip()
        .lower())

## Load Fine-tuned Model

We load the saved checkpoint from `final_model/`.
Using `torch.bfloat16` for efficient inference.

In [3]:
MODEL_PATH = "./final_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="cuda"
)
model.eval()

print(f"Model loaded from {MODEL_PATH}")
print(f"VRAM used: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Model loaded from ./final_model
VRAM used: 0.99 GB


## Inference Function

We run greedy decoding (`do_sample=False`) for reproducibility.
Greedy decoding always picks the highest probability token at each step,
giving deterministic outputs — important for fair benchmarking.

In [4]:
def generate_answer(problem, max_new_tokens=512):
    prompt = f"<|user|>\n{problem}\n<|assistant|>\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Strip prompt tokens from output
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

## Evaluate on MATH Test Split (Algebra)

We use the algebra subset from `EleutherAI/hendrycks_math`, filtered to
Levels 1–3 to match our training distribution.

This gives us the **in-distribution** accuracy — how well the model
learned from the distilled traces.

In [5]:
math_ds = load_dataset("EleutherAI/hendrycks_math", "algebra", split="test")
math_ds = math_ds.filter(lambda x: x['level'] in ['Level 1', 'Level 2', 'Level 3'])
print(f"MATH algebra test samples: {len(math_ds)}")

correct, total = 0, 0
math_results = []

for item in tqdm(math_ds, desc="Evaluating MATH"):
    trace = generate_answer(item['problem'])
    predicted = extract_answer_tag(trace)
    gold = extract_boxed(item['solution'])

    if not predicted or not gold:
        math_results.append({
            "correct": False,
            "predicted": predicted,
            "gold": gold
        })
        total += 1
        continue

    is_correct = normalize(predicted) == normalize(gold)
    if is_correct:
        correct += 1
    total += 1
    math_results.append({
        "problem": item['problem'],
        "predicted": predicted,
        "gold": gold,
        "correct": is_correct,
        "level": item['level']
    })

math_acc = correct / total
print(f"\nMATH algebra pass@1: {math_acc:.2%} ({correct}/{total})")

Filter:   0%|          | 0/1187 [00:00<?, ? examples/s]

MATH algebra test samples: 597


Evaluating MATH: 100%|██████████| 597/597 [10:02<00:00,  1.01s/it]


MATH algebra pass@1: 44.56% (266/597)


## Evaluate on GSM8K

GSM8K is a completely separate dataset of grade school math word problems.
This tests **transfer** — whether the CoT reasoning style generalised
beyond the MATH dataset the model was trained on.

GSM8K answers are formatted differently from MATH — they appear after
a `####` delimiter rather than inside `\boxed{}`.

In [6]:
gsm8k_ds = load_dataset("gsm8k", "main", split="test")
gsm8k_ds = gsm8k_ds.select(range(100))
print(f"GSM8K test samples: {len(gsm8k_ds)}")

def extract_gsm8k_answer(text):
    """GSM8K gold answers appear after #### delimiter e.g. '#### 42'"""
    match = re.search(r'####\s*(\-?[\d,]+)', text)
    if match:
        return match.group(1).replace(',', '').strip()
    return None

correct, total = 0, 0
gsm8k_results = []

for item in tqdm(gsm8k_ds, desc="Evaluating GSM8K"):
    trace = generate_answer(item['question'])
    predicted = extract_answer_tag(trace)
    gold = extract_gsm8k_answer(item['answer'])

    if not predicted or not gold:
        gsm8k_results.append({
            "correct": False,
            "predicted": predicted,
            "gold": gold
        })
        total += 1
        continue

    is_correct = normalize(predicted) == normalize(gold)
    if is_correct:
        correct += 1
    total += 1
    gsm8k_results.append({
        "problem": item['question'],
        "predicted": predicted,
        "gold": gold,
        "correct": is_correct
    })

gsm8k_acc = correct / total
print(f"\nGSM8K pass@1: {gsm8k_acc:.2%} ({correct}/{total})")

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

GSM8K test samples: 100


Evaluating GSM8K: 100%|██████████| 100/100 [01:36<00:00,  1.04it/s]


GSM8K pass@1: 39.00% (39/100)


## Final Results Summary

In [9]:
import os
import json
print("=" * 40)
print("EVALUATION RESULTS — Fine-tuned Model")
print("=" * 40)
print(f"MATH algebra (L1-3):  {math_acc:.2%}")
print(f"GSM8K:                {gsm8k_acc:.2%}")
print("=" * 40)

os.makedirs("results", exist_ok=True)

with open("results/ft_model_eval.json", "w") as f:
    json.dump({
        "math_accuracy": math_acc,
        "gsm8k_accuracy": gsm8k_acc,
        "math_results": math_results,
        "gsm8k_results": gsm8k_results
    }, f, indent=2)

print("Saved to results/ft_model_eval.json")

EVALUATION RESULTS — Fine-tuned Model
MATH algebra (L1-3):  44.56%
GSM8K:                39.00%
Saved to results/ft_model_eval.json
